<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Classification 2  - Training & Validating - CIFAR10

The CIFAR-10 dataset (Canadian Institute For Advanced Research) is a popular benchmark dataset containing 60,000 colour images (50,000 for training and 10,000 for testing).

Each image is 32×32 pixels with 3 RGB channels (unlike MNIST's single grayscale channel), making it significantly more complex.

There are 10 classes which include everyday objects such as aeroplanes, cars, birds, cats, deer, dogs, frogs, horses, ships and trucks, with exactly 6,000 images per class.

Compared to MNIST, CIFAR-10 is considered a harder problem due to its colour images and varied backgrounds.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#Input parameters
batch_size = 4
num_parallel_processors = 4
learning_rate = 0.001
momentum = 0.9
num_epochs = 100

In [ ]:
import torchvision.transforms as transforms

# 2.1. Creating data transformer

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2471, 0.2435, 0.2616))])
print("Data transformer updated for CIFAR-10.")

In [ ]:
import torchvision

# 2.2. Downloading public dataset and applying transformations simultaneously


# Download CIFAR-10 Training Dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)


# Download CIFAR-10 Testing Dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

# Class list
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print("CIFAR-10 datasets downloaded and transformed. Classes defined.")

In [ ]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=num_parallel_processors)


testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=num_parallel_processors)

# functions to show an image
def imshow(img):
    # Unnormalize for CIFAR-10: img = img * std + mean
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2471, 0.2435, 0.2616])
    img = img * std[:, None, None] + mean[:, None, None]
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))


# Get some random training images for each class
class_images = {class_name: [] for class_name in classes}
images_per_class = 2 # Number of images to display per class

for i, data in enumerate(trainloader):
    images, labels = data
    for j in range(len(labels)):
        label = classes[labels[j]]
        if len(class_images[label]) < images_per_class:
            class_images[label].append(images[j])
    # Stop once we have enough images for each class
    if all(len(class_images[class_name]) == images_per_class for class_name in classes):
        break

# Plot images for each class
plt.figure(figsize=(15, 8))
for i, class_name in enumerate(classes):
    for j, img in enumerate(class_images[class_name]):
        ax = plt.subplot(len(classes), images_per_class, i * images_per_class + j + 1)
        imshow(img)
        ax.set_title(class_name)
        ax.axis('off')
plt.tight_layout()
plt.show()

print("Data loaders created and classes updated for CIFAR-10.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

'''
Network arrangement

    Input -> Conv1 -> Relu -> Pool -> Conv2 -> Relu -> Pool -> FC1 -> Relu -> FC2 -> Relu -> FC3 -> Output

'''


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5) # Changed in_channels to 3 for CIFAR-10
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU() # Activation function
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # Corrected for 32x32 CIFAR-10 input: (32-5+1)/2 = 14, (14-5+1)/2 = 5. So, 16*5*5 = 400
        self.fc2 = nn.Linear(120, 84) # In-channels, Out-Channels
        self.fc3 = nn.Linear(84, 10) # In-channels, Out-Channels

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 16 * 5 * 5) # Corrected for 32x32 CIFAR-10 input
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net.to(device)
print(f"Model moved to {device}")

# 1.2 Visualizing network
from torchsummary import summary
print("Network - ")
summary(net, (3, 32, 32)) # Updated input size for CIFAR-10

In [ ]:
import torch.optim as optim
import torch.nn as nn

# 2.3 Creating loss function module
criterion = nn.CrossEntropyLoss()

# 2.4 Creating optimizer module
# learning rate (lr) = 0.001 and momentum = 0.9 are common starting points.
optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=momentum)

# 4. Training
epochs = num_epochs # Number of epochs to train the model

# Initialize list to store training losses
train_losses = []

print("Starting Training...")

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # Move inputs and labels to the specified device (CPU/GPU)
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = net(inputs)
        loss = criterion(outputs, labels)

        # Backward pass + optimize
        loss.backward()
        optimizer.step()

        # Print statistics
        running_loss += loss.item()

    # Calculate average loss for the epoch
    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)

    # Print average loss after each epoch
    print(f'Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.3f}')

print('Finished Training')

In [ ]:
## Training block

# 1. Save the trained model's state dictionary
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)
print(f"Model state dictionary saved to {PATH}")

# 2. Plot the train_losses list
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.grid(False)
plt.show()
print("Training loss plot displayed.")

# 3. Create a new Net instance and load the saved model state dictionary
# The Net class definition is assumed to be accessible from a previous cell (e.g., cell 9811653a)
# import torch.nn as nn # Not needed if Net is already defined and nn is imported globally

new_net = Net()
new_net.load_state_dict(torch.load(PATH))
new_net.to(device) # Move the new model to the same device
new_net.eval() # Set the new model to evaluation mode
print("New model instance created and loaded with saved state dictionary.")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get all predictions and true labels from the test set
all_labels = []
all_predicted = []
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

# Compute the confusion matrix
cm = confusion_matrix(all_labels, all_predicted)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

## Receiver Operating Characteristic (ROC) Curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# Initialize lists to store true labels and predicted probabilities
y_true = []
y_pred_proba = []

# Iterate through the testloader to collect predictions
net.eval() # Set the model to evaluation mode
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = net(images)
        probabilities = torch.softmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred_proba.extend(probabilities.cpu().numpy())

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_proba = np.array(y_pred_proba)

# Create a Matplotlib figure and axes for plotting ROC curves
plt.figure(figsize=(10, 8))

# Plot ROC curve for each class
for i in range(len(classes)):
    # Binarize true labels for the current class
    y_true_binarized = (y_true == i).astype(int)
    # Extract predicted probabilities for the current class
    class_probabilities = y_pred_proba[:, i]

    # Calculate ROC curve and AUC score
    fpr, tpr, _ = roc_curve(y_true_binarized, class_probabilities)
    roc_auc = auc(fpr, tpr)

    # Plot the ROC curve
    plt.plot(fpr, tpr, label=f'ROC curve of class {classes[i]} (area = {roc_auc:.2f})')

# Add a diagonal dashed line for a random classifier
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (area = 0.50)')

# Set labels, title, and legend
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve - CIFAR-10')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


## Next Steps and Potential Improvements:

1.  **Hyperparameter Tuning**: Experiment with different learning rates, batch sizes, and optimizer parameters (e.g., trying Adam optimizer instead of SGD).
2.  **Regularization**: Add techniques like Dropout layers or L2 regularization to the network to prevent overfitting, especially if the model performs significantly better on the training set than on the test set.
3.  **Data Augmentation**: Implement more advanced data augmentation techniques (e.g., random flips, rotations, color jitter) using `torchvision.transforms` to increase the diversity of the training data and improve generalization.
4.  **Early Stopping**: Introduce early stopping during training to stop when the validation loss stops improving, which can prevent overfitting and save computational resources.
5.  **Visualize Misclassifications**: Identify and visualize images that the model misclassifies to understand common failure modes.
6.  **Transfer Learning**: For more complex tasks or smaller datasets, consider using pre-trained models (e.g., from `torchvision.models`) and fine-tuning them for CIFAR-10.
7.  **Advanced Evaluation Metrics**: Beyond the confusion matrix and ROC curve, consider other metrics like Precision, Recall, F1-Score, especially if class imbalance is a concern.